In [ ]:
import numpy as np
import pickle
import sys

from sklearn import linear_model as lm

sys.path.append('/home/austin/Basic')
from utils_np import softplus_inverse,safe_softplus

sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

from sklearn.model_selection import GridSearchCV

In [ ]:
myDict = pickle.load(open('Unbalanced_Elastic_12_enc_1.0.p','rb'))

In [ ]:
myDict.keys()

In [ ]:
S_train = myDict['S_train']
S_test = myDict['S_test']

Si_train = softplus_inverse(S_train)
Si_test = softplus_inverse(S_test)

In [ ]:
fnm='/media/austin/ThickBoy__1/DataAgression_Granger2/Aggression_sub_12.mat'
power,coherence,granger,labels = load_data(fnm,fBounds=(1,56),
                        feature_list=['power','coherence','granger'])

myLabel = labels['windows']
mouse = np.asarray(myLabel['mouse'])
group = np.asarray(myLabel['group'])
expDate = np.asarray(myLabel['expDate'])
behavior = np.asarray(myLabel['behavior'])
behaviornon1 = np.asarray(myLabel['behaviornon1'])
time = np.asarray(myLabel['time'])
condition = np.asarray(myLabel['condition'])
N = len(mouse)

indx_pos = (behaviornon1==1)&(condition==4)
indx_neg1 = (behaviornon1==2)&(condition==4)
indx_neg2 = (behaviornon1==2)&(condition==6)
indx_neg3 = (behaviornon1==2)&(condition==8)
indx_neg = indx_neg1|indx_neg2|indx_neg3
indx_tot = indx_neg|indx_pos

y = np.zeros(N)
y[indx_pos] = 1

mouse = mouse[indx_tot]
group = group[indx_tot]
expDate = expDate[indx_tot]
behavior = behavior[indx_tot]
behaviornon1 = behaviornon1[indx_tot]
time = time[indx_tot]
condition = condition[indx_tot]
y = y[indx_tot]

N = len(mouse)

training_set_idx = np.ones(N)
training_set_idx[mouse=='Mouse048'] = 0
training_set_idx[mouse=='Mouse7980'] = 0
training_set_idx[mouse=='Mouse7998'] = 0

granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6


In [ ]:
X = np.hstack((power,coherence,granger))
X = X[indx_tot]

X_subset = np.hstack((power,coherence))
X_subset = X_subset[indx_tot]

print(mouse.shape)
print(X.shape)

X_train = X[training_set_idx==1]
Xs_train = X_subset[training_set_idx==1]
m_train = mouse[training_set_idx==1]
y_train = y[training_set_idx==1]

X_test = X[training_set_idx==0]
Xs_test = X_subset[training_set_idx==0]
m_test = mouse[training_set_idx==0]
y_test = y[training_set_idx==0]



In [ ]:
parameters={'alpha':[0.0001,0.001,0.01,0.1,1.0,10.0,100.0,1000.0]}
model_lm = lm.ElasticNet(max_iter=100000)
clf = GridSearchCV(model_lm,parameters)
clf.fit(Xs_train,Si_train)

In [ ]:
model_lm = clf.best_estimator_

In [ ]:
model_lm.alpha

In [ ]:
model_lm.l1_ratio

In [ ]:
Spred_train = model_lm.predict(Xs_train)

In [ ]:
np.mean((Si_train-Spred_train)**2)

In [ ]:
Spred_test = model_lm.predict(Xs_test)
np.mean((Si_test-Spred_test)**2)

### Evaluate an alternative regularization setting

In [ ]:
model_lm = lm.ElasticNet(alpha=0.1,l1_ratio=0.9)

In [ ]:
model_lm.fit(Xs_train,Si_train)

In [ ]:
np.mean((Si_train-np.mean(Si_train))**2)

In [ ]:
Spred_train = model_lm.predict(Xs_train)

In [ ]:
np.mean((Si_train-Spred_train)**2)

In [ ]:
np.mean((Si_test-np.mean(Si_test))**2)

In [ ]:
Spred_test = model_lm.predict(Xs_test)

In [ ]:
np.mean((Si_test-Spred_test)**2)

In [ ]:
A_enc_new = model_lm.coef_
B_enc_new = model_lm.intercept_

In [ ]:
from scipy.io import savemat

In [ ]:
type(A_enc_new)

In [ ]:
type(B_enc_new)

In [ ]:
myDict2 = {}
myDict2['A_enc_new'] = A_enc_new
myDict2['B_enc_new'] = B_enc_new

savemat('Reduced_encoder.mat',myDict2)